# Day 5 | Lab 5.1: Advanced Retrieval — Hybrid Search, HyDE, Re-Ranking & Contextual Chunks

**Duration:** ~1.5 hours · **Type:** Dense (with Interview Preparation)

**Why this lab.** Vanilla similarity-search RAG often plateaus around 60–70 % retrieval accuracy on real-world corpora. This lab covers the four moves that consistently push that number higher in production:

- **Hybrid retrieval** — combine BM25 (keyword) with vector search; fixes the cases where pure semantics misses an exact-match keyword.
- **HyDE (Hypothetical Document Embeddings)** — let the LLM imagine an answer first, then embed *that* and retrieve against it; closes the question-vs-answer vocabulary gap.
- **Cross-Encoder Re-Ranking** — pull a wide first-stage candidate set from a fast retriever, then re-score the top N with a slower-but-sharper cross-encoder that reads (query, passage) jointly.
- **Contextual chunks** (Anthropic, 2024) — augment each chunk with a one-line summary of where it sits in the source document, so retrieval doesn't lose section context.

**Learning objectives.** By the end of this lab, you should be able to:

1. Build similarity, MMR, BM25, and hybrid (BM25 + vector) retrievers and explain when each wins.
2. Implement HyDE end-to-end without a custom retriever class.
3. Wrap any retriever in a cross-encoder `ContextualCompressionRetriever`.
4. Explain the contextual-chunk pattern from Anthropic's 2024 blog post and decide when it's worth the extra embedding-time cost.
5. Pick the right strategy for a given corpus + query mix.

**Scenario.** A research-knowledge corpus combining Wikipedia summaries with arXiv ML/CV/NLP papers (transformers, ResNet, BERT, attention mechanisms) — mixed-format, exactly the kind of heterogeneous knowledge base where naive top-k similarity falls short.

**Tools.** LangChain v1 · `langchain-community` (BM25, FAISS) · `langchain.retrievers` (`EnsembleRetriever`, `ContextualCompressionRetriever`) · `sentence-transformers` (cross-encoder) · OpenAI `text-embedding-3-small` · GPT-5-mini for answers and Claude Sonnet 4.5 for HyDE generation.

Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)

---


## 1. Install Dependencies


In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install -q 'langchain>=1.0' 'langchain-core>=1.0' 'langchain-openai>=1.0' 'langchain-anthropic>=0.3'
# !pip install -q 'langchain-community' 'langchain-text-splitters' 'langchain-chroma'
# !pip install -q faiss-cpu rank-bm25 'sentence-transformers>=3' pymupdf jq gdown


## 2. API Key Configuration


In [1]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv("..\\.env")
except ImportError:
    pass

for key in ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


OPENAI_API_KEY: ✅ Loaded
ANTHROPIC_API_KEY: ✅ Loaded


---
## 3. Load and Prepare the Document Corpus

We'll use a public mixed-format corpus: ~50 Wikipedia summaries (`wikidata_rag_demo.jsonl`) plus 5 arXiv PDFs covering CNN/ResNet/transformer/BERT/Vision-Transformer papers. The mix matters — short Wikipedia articles favour vector similarity; technical PDFs with named entities (model names, layer types) favour BM25. This is exactly the kind of corpus where hybrid retrieval pays off.


In [4]:
# Download the dataset
!gdown 1aZxZejfteVuofISodUrY2CDoyuPLYDGZ
!unzip -o rag_docs.zip

Downloading...
From: https://drive.google.com/uc?id=1aZxZejfteVuofISodUrY2CDoyuPLYDGZ
To: /content/rag_docs.zip
100% 5.92M/5.92M [00:00<00:00, 59.6MB/s]
Archive:  rag_docs.zip
   creating: rag_docs/
  inflating: rag_docs/attention_paper.pdf  
  inflating: rag_docs/cnn_paper.pdf  
  inflating: rag_docs/resnet_paper.pdf  
  inflating: rag_docs/vision_transformer.pdf  
  inflating: rag_docs/wikidata_rag_demo.jsonl  


In [5]:
import json
from glob import glob
from langchain_community.document_loaders import JSONLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter # Corrected import path
from langchain_core.documents import Document # Corrected import path for Document

# --- Load Wikipedia articles from JSONL ---
loader = JSONLoader(
    file_path='./rag_docs/wikidata_rag_demo.jsonl',
    jq_schema='.', text_content=False, json_lines=True
)
wiki_raw = loader.load()

wiki_docs = []
for doc in wiki_raw:
    data = json.loads(doc.page_content)
    content = ' '.join(data['paragraphs'])
    metadata = {"title": data['title'], "id": data['id'], "source": "Wikipedia"}
    wiki_docs.append(Document(page_content=content, metadata=metadata))

print(f"Wikipedia articles: {len(wiki_docs)}")

# --- Load and chunk PDF documents ---
splitter = RecursiveCharacterTextSplitter(chunk_size=3500, chunk_overlap=500)
pdf_files = sorted(glob('./rag_docs/*.pdf'))
paper_docs = []
for fp in pdf_files:
    pages = PyMuPDFLoader(fp).load()
    chunks = splitter.split_documents(pages)
    paper_docs.extend(chunks)
    print(f"  {fp.split('/')[-1]}: {len(pages)} pages → {len(chunks)} chunks")

print(f"\nPDF chunks: {len(paper_docs)}")

# --- Combine into unified corpus ---
total_docs = wiki_docs + paper_docs
print(f"Total corpus: {len(total_docs)} documents")

Wikipedia articles: 1801
  attention_paper.pdf: 15 pages → 16 chunks
  cnn_paper.pdf: 11 pages → 11 chunks
  resnet_paper.pdf: 12 pages → 24 chunks
  vision_transformer.pdf: 22 pages → 28 chunks

PDF chunks: 79
Total corpus: 1880 documents


---
## 4. Build the FAISS Vector Index

Embed every chunk with `text-embedding-3-small` (1536-dim, $0.02 / 1M tokens) and store in an in-memory FAISS index. FAISS is the right choice for a few-hundred-document lab — for production scale (>100K docs), switch to a managed store (Chroma, Pinecone, Weaviate, or Bedrock KB).

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize embeddings
embed_model = OpenAIEmbeddings(model='text-embedding-3-small')

# Build FAISS index (takes ~30-60 seconds)
faiss_db = FAISS.from_documents(documents=total_docs, embedding=embed_model)

print(f"✅ FAISS index built: {faiss_db.index.ntotal} vectors, {faiss_db.index.d} dimensions")

✅ FAISS index built: 1880 vectors, 1536 dimensions


---
## 5. Three Baseline Retrievers — Similarity, MMR, Hybrid (BM25 + Vector)

These three are the bread-and-butter retrievers — every advanced strategy in this lab is layered on top of one of them.

| Strategy | How It Works | Strengths | Weaknesses |
|---|---|---|---|
| **Similarity** | Cosine top-K nearest vectors | Simple, fast, accurate for focused queries | Returns redundant near-duplicates |
| **MMR** | Greedy selection balancing relevance with diversity (`lambda`) | Reduces redundancy, broader coverage | Slightly slower; lambda needs tuning |
| **Hybrid (BM25 + Vector)** | Combines keyword (BM25) and semantic via Reciprocal Rank Fusion | Catches exact terms AND meaning | Two indices to maintain; weight tuning |

**Note on the `EnsembleRetriever` import.** The original notebook imported it from `langchain_classic.retrievers.ensemble`. As of LangChain v1, the canonical path is `from langchain.retrievers import EnsembleRetriever` (it re-exports from `langchain_community`). We use the v1 path below.

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

K = 5  # top-K for all retrievers

# --- Strategy 1: Baseline Similarity Search ---
similarity_retriever = faiss_db.as_retriever(
    search_type='similarity',
    search_kwargs={'k': K},
)

# --- Strategy 2: MMR (Maximum Marginal Relevance) ---
mmr_retriever = faiss_db.as_retriever(
    search_type='mmr',
    search_kwargs={
        'k': K,
        'fetch_k': 20,        # candidate pool size
        'lambda_mult': 0.5,   # 1.0 = pure relevance, 0.0 = pure diversity
    },
)

# --- Strategy 3: Hybrid (BM25 + Vector) via Reciprocal Rank Fusion ---
bm25_retriever = BM25Retriever.from_documents(total_docs)
bm25_retriever.k = K

faiss_vector_retriever = faiss_db.as_retriever(
    search_type='similarity',
    search_kwargs={'k': K},
)

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_vector_retriever],
    weights=[0.4, 0.6],  # 40% keyword, 60% semantic
)

print('✅ Three retrievers initialized:')
print(f'   1. Similarity Search    (top-{K})')
print(f'   2. MMR                  (top-{K} from 20 candidates, λ=0.5)')
print(f'   3. Hybrid BM25+Vector   (top-{K}, weights: BM25=0.4, Vector=0.6)')


---
## 6. Side-by-Side Retrieval Comparison

Run the same query through all three retrievers and compare *which* chunks each returns — the overlap, the unique-to-each-strategy chunks, and the diversity of sources. This is the right way to debug retrieval: when answer quality drops, look at the chunks first.

In [8]:
def get_doc_signature(doc):
    """Create a short signature for a document chunk."""
    title = doc.metadata.get('title', '')
    source = doc.metadata.get('source', doc.metadata.get('file_path', 'PDF'))
    if title:
        return title[:50]
    return source.split('/')[-1][:50] if '/' in str(source) else str(source)[:50]


def compare_retrievers_side_by_side(query, retrievers_dict, top_n=5):
    """Run a query through multiple retrievers and display a comparison table."""
    print(f"\n{'='*80}")
    print(f"  QUERY: \"{query}\"")
    print(f"{'='*80}\n")

    all_results = {}
    for name, retriever in retrievers_dict.items():
        docs = retriever.invoke(query)
        all_results[name] = docs
        sources = [get_doc_signature(d) for d in docs[:top_n]]
        print(f"  {name}:")
        for i, src in enumerate(sources, 1):
            print(f"    {i}. {src}")
        print()

    # Overlap analysis
    sets = {name: set(get_doc_signature(d) for d in docs)
            for name, docs in all_results.items()}
    names = list(sets.keys())
    if len(names) >= 2:
        overlap_all = sets[names[0]].intersection(*[sets[n] for n in names[1:]])
        unique_per = {n: sets[n] - sets[names[0]].union(*[sets[m] for m in names if m != n])
                      for n in names}
        print(f"  📊 OVERLAP ANALYSIS:")
        print(f"     Common across ALL strategies: {len(overlap_all)} docs")
        for n in names:
            print(f"     Unique to {n}: {len(unique_per[n])} docs")

    return all_results

In [12]:
# --- Query 4: NLP-specific query ---
results_4 = compare_retrievers_side_by_side(
    "word sense disambiguation in natural language processing",
    retrievers
)


  QUERY: "word sense disambiguation in natural language processing"

  Similarity:
    1. Natural language processing
    2. attention_paper.pdf
    3. attention_paper.pdf
    4. vision_transformer.pdf
    5. attention_paper.pdf

  MMR:
    1. Natural language processing
    2. vision_transformer.pdf
    3. Semantic Web
    4. attention_paper.pdf
    5. Deep learning

  Hybrid (BM25+Vec):
    1. Natural language processing
    2. attention_paper.pdf
    3. attention_paper.pdf
    4. vision_transformer.pdf
    5. attention_paper.pdf

  📊 OVERLAP ANALYSIS:
     Common across ALL strategies: 3 docs
     Unique to Similarity: 0 docs
     Unique to MMR: 2 docs
     Unique to Hybrid (BM25+Vec): 4 docs


---
## 7. BM25 Deep Dive — When Keywords Beat Semantics

BM25 wins when the query contains **named entities, product codes, or rare technical terms** that the embedding model may have averaged into a generic vector. Test it on queries with specific model names and architecture terms below.

In [15]:
# Direct comparison: BM25-only vs Vector-only on keyword-heavy queries
keyword_heavy_queries = [
    "CNN architecture layers",
    "BERT pre-training masked language model",
    "attention mechanism query key value",
    "backpropagation gradient descent",
]

print("BM25 vs. Vector Search — Keyword-Heavy Queries")
print("=" * 75)

for query in keyword_heavy_queries:
    bm25_docs = bm25_retriever.invoke(query)
    vec_docs = similarity_retriever.invoke(query)

    bm25_sources = set(get_doc_signature(d) for d in bm25_docs)
    vec_sources = set(get_doc_signature(d) for d in vec_docs)

    only_bm25 = bm25_sources - vec_sources
    only_vec = vec_sources - bm25_sources
    both = bm25_sources & vec_sources

    print(f"\n  Query: \"{query}\"")
    print(f"    Common:     {len(both)} docs")
    print(f"    BM25 only:  {len(only_bm25)} docs  {list(only_bm25)[:2]}")
    print(f"    Vector only: {len(only_vec)} docs  {list(only_vec)[:2]}")

BM25 vs. Vector Search — Keyword-Heavy Queries

  Query: "CNN architecture layers"
    Common:     1 docs
    BM25 only:  1 docs  ['resnet_paper.pdf']
    Vector only: 1 docs  ['CNN']

  Query: "BERT pre-training masked language model"
    Common:     1 docs
    BM25 only:  0 docs  []
    Vector only: 1 docs  ['attention_paper.pdf']

  Query: "attention mechanism query key value"
    Common:     1 docs
    BM25 only:  1 docs  ['vision_transformer.pdf']
    Vector only: 1 docs  ['Session key']

  Query: "backpropagation gradient descent"
    Common:     2 docs
    BM25 only:  2 docs  ['Silva Kaputikyan', 'cnn_paper.pdf']
    Vector only: 0 docs  []


## 8. HyDE — Hypothetical Document Embeddings

A retrieval failure mode you'll see often: the **user's question** and the **answer text in the corpus** use very different vocabulary. The question asks "How does multi-head attention work?" but the relevant chunk says "Each attention head computes scaled dot-product attention with its own learned projection matrices…". Pure-question embeddings can miss it.

**HyDE** (Gao et al., 2022) is a one-trick fix:

1. Send the user's question to an LLM and ask for a **hypothetical answer** (the LLM doesn't have the corpus — that's fine).
2. **Embed the hypothetical answer**, not the question.
3. Use that embedding for retrieval.

Why it works: the hypothetical answer lands in the same vocabulary neighbourhood as real answers in the corpus. You're matching answer-shaped vectors against answer-shaped chunks.

**Cost:** one extra LLM call per query (small, often Haiku-class), no extra embeddings on the corpus side. **Latency:** +200–600 ms.

**When it backfires:** highly factual lookups where the LLM hallucinates the wrong fact in its hypothetical, dragging retrieval off-target. Best suited to *concept-explanation* questions, not *fact-lookup* questions.


In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

hyde_llm = ChatAnthropic(model='claude-sonnet-4-5-20250929', temperature=0)

hyde_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are an expert technical writer. Given a question, write a single-paragraph '
               'plausible passage that would answer it. Make it factual-sounding even if you have '
               'to make it up. Output ONLY the passage, no preamble.'),
    ('human', 'Question: {question}\n\nHypothetical answer passage:')
])

hyde_chain = hyde_prompt | hyde_llm

def generate_hypothetical(question: str) -> str:
    """Generate a hypothetical answer paragraph to use as the retrieval query."""
    return hyde_chain.invoke({'question': question}).content

print('✅ HyDE generator ready (Claude Sonnet 4.5)')


In [ ]:
# Generate a hypothetical answer for one of our test queries
hyde_query = 'How does ResNet improve upon standard CNNs with skip connections?'
hypothetical = generate_hypothetical(hyde_query)

print('Original question:')
print(f'  {hyde_query}\n')
print('Hypothetical answer (used as the retrieval query):')
print(hypothetical)


In [ ]:
from langchain_openai import OpenAIEmbeddings
import numpy as np

# Embed both the vanilla query and the hypothetical answer with the SAME model
# the corpus was indexed with
embed_model = OpenAIEmbeddings(model='text-embedding-3-small')
vanilla_vec = embed_model.embed_query(hyde_query)
hyde_vec    = embed_model.embed_query(hypothetical)

# Cosine distance between the two embedding regions — bigger means HyDE is
# pulling us into a meaningfully different part of vector space
v1, v2 = np.array(vanilla_vec), np.array(hyde_vec)
cos_sim = float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))
print(f'Cosine similarity between vanilla query and HyDE query embeddings: {cos_sim:.3f}')
print('(closer to 1.0 → HyDE not changing much; lower → HyDE moving the search region)')


In [ ]:
# Use the embedding directly via FAISS — this is the HyDE retrieval step
vanilla_docs = faiss_db.similarity_search_by_vector(vanilla_vec, k=5)
hyde_docs    = faiss_db.similarity_search_by_vector(hyde_vec,    k=5)

vanilla_sigs = [get_doc_signature(d) for d in vanilla_docs]
hyde_sigs    = [get_doc_signature(d) for d in hyde_docs]

print('Vanilla query → top-5 sources:')
for i, s in enumerate(vanilla_sigs, 1):
    print(f'  {i}. {s}')

print('\nHyDE query → top-5 sources:')
for i, s in enumerate(hyde_sigs, 1):
    print(f'  {i}. {s}')

overlap = set(vanilla_sigs) & set(hyde_sigs)
print(f'\nOverlap: {len(overlap)} docs   |   New from HyDE: {len(set(hyde_sigs) - set(vanilla_sigs))}')


**HyDE failure mode to remember.** If you ask HyDE about a topic the corpus genuinely doesn't cover (e.g. a brand-new product or a 2026 regulation), the LLM will write a confident-sounding but fictional passage, and you'll retrieve chunks that are 'similar to the fiction' rather than 'similar to the truth'. Vanilla retrieval would return zero close matches and you'd know to back off. HyDE *hides* the no-coverage case — guard against it with a similarity-score threshold.

---
## 9. Cross-Encoder Re-Ranking

**Bi-encoder (what your retriever uses).** Embeds query and chunks **separately**, compares with cosine. Fast (you can pre-index), but weaker — the model never sees query and chunk together.

**Cross-encoder (what we add now).** Takes `(query, chunk)` as a single input pair, runs them through a transformer, outputs a relevance score. Much more accurate, but ~100x slower per pair — you can't pre-index, you must score at query time.

**The pattern.** Retrieve top-N (e.g. 20) cheaply with the bi-encoder, then re-rank with the cross-encoder and keep top-K (e.g. 5). LangChain wraps this as `ContextualCompressionRetriever` with a `CrossEncoderReranker` compressor.

**Latency budget.** A small cross-encoder (`ms-marco-MiniLM-L-6-v2`, 22M params) scores 20 pairs in ~100ms on CPU. The retrieval-quality lift is usually worth it for accuracy-critical workflows.

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker

# Load a small, fast cross-encoder fine-tuned on MS-MARCO passage ranking
# (~22M params; ~100MB download on first use; runs on CPU fine).
cross_encoder = HuggingFaceCrossEncoder(model_name='cross-encoder/ms-marco-MiniLM-L-6-v2')
reranker = CrossEncoderReranker(model=cross_encoder, top_n=5)

# Step 1: cheap bi-encoder retrieval pulls a wider candidate pool (20 docs)
# Step 2: cross-encoder re-ranks to keep the top-5
wide_retriever = faiss_db.as_retriever(search_kwargs={'k': 20})

rerank_retriever = ContextualCompressionRetriever(
    base_retriever=wide_retriever,
    base_compressor=reranker,
)
print('✅ Cross-encoder re-ranker wired up (retrieve top-20 → rerank to top-5)')


In [ ]:
import time

rerank_query = 'How is self-attention used in transformers for NLP and computer vision?'

# Time vanilla similarity vs cross-encoder re-ranked retrieval
t0 = time.perf_counter()
vanilla = similarity_retriever.invoke(rerank_query)
vanilla_ms = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
reranked = rerank_retriever.invoke(rerank_query)
rerank_ms = (time.perf_counter() - t0) * 1000

print(f'Vanilla similarity:        {vanilla_ms:6.1f} ms   {len(vanilla)} docs')
print(f'Wide retrieval + rerank:   {rerank_ms:6.1f} ms   {len(reranked)} docs')
print(f'Re-rank latency overhead:  {rerank_ms - vanilla_ms:+6.1f} ms')

print('\nVanilla top-5:')
for i, d in enumerate(vanilla, 1):
    print(f'  {i}. {get_doc_signature(d)}')
print('\nRe-ranked top-5:')
for i, d in enumerate(reranked, 1):
    print(f'  {i}. {get_doc_signature(d)}')


In [ ]:
# Retrieval-quality lift: how often does the re-ranker change the top-3?
rerank_eval_queries = [
    'What is the difference between BERT and GPT?',
    'How does ResNet handle the vanishing gradient problem?',
    'What does multi-head attention compute?',
    'How does word2vec learn word embeddings?',
    'What is the role of positional encoding in transformers?',
]

changes = 0
for q in rerank_eval_queries:
    v_top3 = {get_doc_signature(d) for d in similarity_retriever.invoke(q)[:3]}
    r_top3 = {get_doc_signature(d) for d in rerank_retriever.invoke(q)[:3]}
    diff = len(v_top3.symmetric_difference(r_top3))
    if diff > 0:
        changes += 1
    print(f'  "{q[:55]}..." → {diff} doc(s) swapped in top-3')

print(f'\nRe-ranker changed the top-3 on {changes}/{len(rerank_eval_queries)} queries')


**When is re-ranking worth the latency?** A rule of thumb:
- **High-value, low-volume queries** (legal/medical/regulatory Q&A) → always rerank.
- **Conversational chatbots** with strict latency SLAs → use rerank only when retrieval-quality metrics (Hit Rate@K, MRR — see Lab 5.4) show a meaningful lift on your eval set.
- **Bulk pipelines** (batch enrichment, document tagging) → rerank, latency doesn't matter.

---
## 10. Strategy Selection — Quick Decision Reference

| Workload | Recommended retriever | Why |
|---|---|---|
| Small corpus (<1K docs), focused queries | Similarity | Simple, fast, accurate enough |
| Many near-duplicate chunks | MMR | Diversity prevents redundant context |
| Mix of natural-language + named-entity queries | **Hybrid (BM25 + vector)** | Best general-purpose default |
| Open-ended conceptual queries with vocabulary mismatch | Multi-Query | LLM rewrites widen the recall net |
| Question-shaped queries against answer-shaped chunks | HyDE | Aligns query and chunk in answer-space |
| Accuracy-critical, latency-tolerant | **Hybrid + Cross-Encoder Re-Rank** | Highest precision @top-K |
| Large corpus (>100K docs), low-latency SLA | Similarity (FAISS HNSW) + cached re-rank | ANN scales; only rerank top-50 |

**Costs at this lab's scale (~200 chunks, ~16 RAG calls, ~16 judge calls):** under $0.10 per full run. The cross-encoder is local — no API cost. Multi-query adds one rewrite call per query (~$0.0005). HyDE adds one Claude generation per query (~$0.003).

**Decision shorthand:** start with **hybrid** (BM25 + vector via `EnsembleRetriever`) as your default; add **HyDE** when questions and corpus vocabulary diverge; add a **cross-encoder re-ranker** when latency budget allows and precision matters more than throughput; consider **contextual chunks** only when retrieval failures cluster around chunks that need section context to make sense.

The full decision matrix is in the table above; in the wild, most teams ship with hybrid + a cheap re-ranker and revisit only if recall measurements drop.


## 11. Contextual Chunks (Anthropic, 2024)

A different angle on the same problem. Pure-vector retrieval struggles when a chunk is meaningful only in the context of its surrounding document — e.g., a chunk that says "This applies only to fixed-rate accounts." is useless without knowing which section it's from.

**Anthropic's contextual-chunks pattern** (Sept 2024) prepends a one-line context summary to each chunk *before* embedding. The summary is generated by an LLM that sees both the chunk and the entire source document.

```
context: "This is a clause in the 'Loan Pre-payment Terms' section of the 2024 Mortgage Policy."
chunk:   "Borrowers may pre-pay up to 25 % of the outstanding principal annually..."
```

The combined string is what gets embedded. The downstream retrieval call doesn't change at all — only the corpus prep step is different.

**Reported gains** (Anthropic's blog): 35 % reduction in retrieval failures on their internal datasets. Costs: one LLM call per chunk at ingest time; if you're using Claude with prompt caching, the per-chunk marginal cost drops to roughly 1¢ per 100 chunks.

This section walks through one small example. Lab 5.4 (RAG Evaluation) is where you'd actually measure the lift on your own corpus.


In [5]:
from langchain_anthropic import ChatAnthropic

context_llm = ChatAnthropic(model='claude-sonnet-4-5-20250929', temperature=0)


In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

def generate_chunk_context(document: str, chunk: str) -> str:
    """Ask Claude to write a 3-4 sentence situational context for a chunk.

    Args:
        document: Full text of the parent document.
        chunk:    The specific chunk we want to situate.

    Returns:
        A short context string to prepend to the chunk before embedding.
    """
    chunk_process_prompt = (
        "You are an AI assistant specializing in research-paper analysis. "
        "Your task is to provide brief, relevant context for a chunk of text "
        "based on the following research paper.\n\n"
        "<paper>\n{paper}\n</paper>\n\n"
        "<chunk>\n{chunk}\n</chunk>\n\n"
        "Provide a concise context (3-4 sentences max) for this chunk:\n"
        "- Situate it within the overall document for the purposes of improving search retrieval.\n"
        "- Answer ONLY with the succinct context, no preamble, no quoting.\n"
    )
    prompt_template = ChatPromptTemplate.from_template(chunk_process_prompt)
    chain = prompt_template | context_llm | StrOutputParser()
    return chain.invoke({'paper': document, 'chunk': chunk})


In [7]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

def create_contextual_chunks(file_path: str, chunk_size: int = 3500, chunk_overlap: int = 0) -> list[Document]:
    """Load a PDF, split into chunks, and prepend an LLM-generated context to each."""
    print(f'Loading pages: {file_path}')
    doc_pages = PyMuPDFLoader(file_path).load()

    print(f'Chunking pages: {file_path}')
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    doc_chunks = splitter.split_documents(doc_pages)

    print(f'Generating contextual chunks: {file_path}')
    # Reassemble the full document so every chunk gets the same parent context
    original_doc = '\n'.join(d.page_content for d in doc_chunks)

    contextual_chunks: list[Document] = []
    for chunk in doc_chunks:
        chunk_metadata_upd = {
            'id':     str(uuid.uuid4()),
            'page':   chunk.metadata['page'],
            'source': chunk.metadata['source'],
            'title':  chunk.metadata['source'].split('/')[-1],
        }
        context = generate_chunk_context(original_doc, chunk.page_content)
        contextual_chunks.append(Document(
            page_content=context + '\n' + chunk.page_content,
            metadata=chunk_metadata_upd,
        ))

    print(f'Finished: {file_path}')
    return contextual_chunks

def create_naive_chunks(file_path: str, chunk_size: int = 3500, chunk_overlap: int = 0) -> list[Document]:
    """Same loader/splitter, but NO context-generation step. Used as the baseline for the benchmark."""
    doc_pages = PyMuPDFLoader(file_path).load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = splitter.split_documents(doc_pages)
    naive_chunks: list[Document] = []
    for chunk in chunks:
        meta = {
            'id':     str(uuid.uuid4()),
            'page':   chunk.metadata['page'],
            'source': chunk.metadata['source'],
            'title':  chunk.metadata['source'].split('/')[-1],
        }
        naive_chunks.append(Document(page_content=chunk.page_content, metadata=meta))
    return naive_chunks


In [8]:
from glob import glob

pdf_files = sorted(glob('./rag_docs/*.pdf'))
print(f'Found {len(pdf_files)} PDFs to process')
for fp in pdf_files:
    print(f'  - {fp}')


Found 0 PDFs to process


**When Is Contextual Retrieval Worth the Embedding-Time Cost?**


**The cost equation.** Contextual retrieval adds **one LLM call per chunk** at indexing time. For a corpus of 10K chunks averaging 3500 chars (~900 tokens), that's:
- ~10K calls × ~3000 input tokens (paper + chunk + prompt) + ~150 output tokens.
- At Claude Sonnet 4.5 pricing (~$3 per 1M input, ~$15 per 1M output), that's about **$90** to index the corpus once. With prompt-caching on the document body, you can cut this by ~60%.
- Inference cost is unchanged — same retriever shape, same answer LLM.

**When it's worth it (✅).**

| Signal | Why contextual helps |
|---|---|
| Long documents (>20 pages) split into many chunks | Each chunk loses parent context; contextualization restores it |
| High-value queries (legal, regulatory, medical, financial) | The 30%+ retrieval-failure reduction directly translates to safer answers |
| Stable, low-churn corpus (re-indexing is rare) | One-time indexing cost amortized over millions of queries |
| Domain with ambiguous chunk content (numbers without units, references to unnamed figures) | Context resolves the ambiguity that vanilla embedding can't |

**When it's NOT worth it (❌).**

| Signal | Why naive is fine |
|---|---|
| Short, self-contained docs (<1 page each) | No parent context to add — the chunk IS the doc |
| High-churn corpus (re-indexed daily, e.g. news feeds) | Embedding-time cost recurs every day; budget kills the ROI |
| Free-text consumer chatbot with $0 SLA on retrieval quality | Lift is real but probably not worth the cost |
| Already-good baseline (Hit Rate@5 > 0.9 on your eval) | Diminishing returns — spend the budget on re-ranking instead |

**Hybrid pattern most teams settle on.** Contextual retrieval for the *high-value* portion of the corpus (regulatory filings, contracts) + naive chunking for the *bulk* portion (changelogs, routine docs). Different collections, different cost profiles, one router on top.

## 12. Conclusion & Key Takeaways

| Concept | What you took away |
|---|---|
| **Baseline retrievers** | Similarity (vector), MMR (vector + diversity), BM25 (keyword), Hybrid (`EnsembleRetriever`). Hybrid is the production default for prose corpora. |
| **HyDE** | One extra LLM call per query closes the question-vs-answer vocabulary gap. Best for concept questions; can backfire on facts. |
| **Cross-Encoder Re-Ranking** | Wrap any retriever in `ContextualCompressionRetriever` with `cross-encoder/ms-marco-MiniLM-L-6-v2` (or larger). Adds ~100–300 ms per query, often worth it. |
| **Contextual Chunks** | Anthropic's 2024 pattern — prepend an LLM-generated context line to each chunk before embedding. ~35 % retrieval-failure reduction in their reports. |
| **Strategy selection** | Default to hybrid + HyDE + (optional) re-ranker. Add contextual chunks only when retrieval failures cluster on context-dependent chunks. |

**What we deliberately did NOT cover** (look for them later):
- MMR-λ and hybrid-weight grid sweeps — niche tuning, mention only.
- LLM-as-Judge scoring of final answers — Lab 5.4 (RAGAS).
- Agentic RAG with multi-step citation extraction — Module 7 (LangGraph).
- 20-question retrieval benchmarks — Lab 5.4 (RAGAS).

**Where this lab sits in the program:**
- Lab 4.3 introduced the canonical LCEL RAG chain.
- This lab upgrades the **retriever** in that chain.
- Lab 5.3 introduces an alternative framework (LlamaIndex) with its own retrieval primitives.
- Lab 5.4 measures how much each retrieval upgrade actually helps via RAGAS.

---


## 13. Stretch Exercise (Optional)

Pick one of these — each takes 20–40 minutes.

1. **Switch the embedding model to Bedrock Titan v2.** Re-run the side-by-side comparison. Do the top-3 chunks change for the same questions? Why or why not?
2. **Tune the cross-encoder cap.** Try `top_n=3, 5, 10, 20` on the re-ranker. Plot or tabulate the trade-off between latency and answer quality. Where's the elbow?
3. **HyDE on a fact-lookup query.** Pick three deliberate fact-lookup questions ("What year was BERT released?", "How many parameters does ResNet-50 have?") and compare HyDE vs vanilla retrieval. Does HyDE make things worse?
4. **Contextual chunks on a structured doc.** Take a 10-page PDF with clear sections. Build two indices (naive vs contextual). Run 10 questions where the answer depends on knowing the section. Measure the improvement.
5. **Re-ranker swap.** Try `BAAI/bge-reranker-large` instead of `ms-marco-MiniLM-L-6-v2`. Compare quality and latency.

---


## Interview Preparation

The questions below mirror what client interviewers commonly ask about retrieval. Use the hint to think through the answer first; use the sketch only to verify your reasoning.

---

**Q1. When does BM25 beat dense semantic retrieval?**

*Hint:* Think about exact-match keywords, rare proper nouns, code identifiers.

*Answer sketch:* When the query contains rare exact-match tokens (product codes, error messages, person/company names) that don't have a strong semantic representation. BM25 indexes the literal token; dense vectors smear the meaning across nearby concepts and can miss the exact hit.

---

**Q2. Why does `EnsembleRetriever` (BM25 + vector) usually beat either alone?**

*Hint:* Think about what each retriever's failure modes look like.

*Answer sketch:* BM25 is precise but recall-fragile (misses paraphrases). Vector is recall-robust but precision-soft on rare keywords. The two failure modes are largely uncorrelated, so the ensemble (RRF or weighted union) recovers most of each method's misses without inheriting the other's weaknesses.

---

**Q3. Walk through the HyDE algorithm in three lines.**

*Hint:* Where does the LLM call happen, and what gets embedded?

*Answer sketch:* (1) Send user question to an LLM, ask for a hypothetical answer. (2) Embed the hypothetical answer (not the question). (3) Run nearest-neighbour search with that embedding against the corpus index. The hypothetical lives in answer-vocabulary space, which matches real answers in the corpus better than question-vocabulary embeddings do.

---

**Q4. When does HyDE backfire?**

*Hint:* What kind of question gives the LLM the most freedom to hallucinate?

*Answer sketch:* Pure fact-lookup questions where the LLM may invent the wrong fact in the hypothetical, pulling retrieval toward the wrong chunks. HyDE is a vocabulary-bridging trick, not a knowledge trick — when the question is "what's the answer," not "what does this concept mean," vanilla retrieval is often safer.

---

**Q5. What does a cross-encoder do that a bi-encoder embedding model doesn't?**

*Hint:* Compare what each model sees at scoring time.

*Answer sketch:* A bi-encoder embeds query and passage independently — the model never sees them together. A cross-encoder takes (query, passage) as a single concatenated input and outputs one relevance score, so it can attend to fine-grained interactions. Slower per pair but much sharper. Used as a re-ranker on a top-N candidate set, never as a first-stage retriever.

---

**Q6. Why pull a wide candidate set from a fast retriever and re-rank, rather than just running the cross-encoder over the whole corpus?**

*Hint:* Latency math at corpus scale.

*Answer sketch:* The cross-encoder is O(N) per query — running it over a 1 M-chunk corpus is multiple seconds at best. The two-stage pattern is: fast retriever returns top-100 in <50 ms, cross-encoder re-scores those 100 in ~200 ms. End-to-end you get cross-encoder-quality top-3 at retriever-class latency.

---

**Q7. What problem do contextual chunks solve, and what does the cost look like?**

*Hint:* Think about a chunk that's meaningless without its section context.

*Answer sketch:* Chunks that depend on their surrounding context (section headers, prior paragraphs, "this clause" references) embed to neighbourhoods that don't reflect their real meaning. Anthropic's 2024 fix prepends a 1-line LLM-generated context summary before embedding. Cost is one LLM call per chunk at ingest time — with prompt caching, ~1¢ per 100 chunks.

---

**Q8. If you were starting from a vanilla similarity-search RAG today and had to pick one upgrade, what would it be?**

*Hint:* Cost vs benefit, biggest needle-mover.

*Answer sketch:* Hybrid retrieval (`EnsembleRetriever` with BM25 + vector). It's a one-line code change, no extra inference cost, and recovers most of the keyword-precision gap that pure vector search has. HyDE and re-ranking layer on top of hybrid; contextual chunks only matter for context-dependent corpora. Hybrid first, always.
